# 6.24 Transfer learning & fine-tuning

Fine-tuning starts from features learned elsewhere, then spends gradient budget carefully on the new task.

A pretrained base can be reused, but the head and base should move at different rates on the new task. Save a copy to Drive to edit.

In [ ]:

import math
import time
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

np.random.seed(42)


def clf_digits_ladder():
    """D1 XOR -> D2 blobs -> D3 noisy moons -> D4 digits -> D5 noisy digits."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def pad_to_64(X):
    X = np.asarray(X, dtype=float)
    if X.shape[1] == 64:
        return X.copy()
    if X.shape[1] > 64:
        return X[:, :64].copy()
    out = np.zeros((X.shape[0], 64), dtype=float)
    out[:, :X.shape[1]] = X
    return out


def split_scale(X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te




def softmax_logits(X, W):
    logits = X @ W
    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def train_small_mlp_loss(X, y, random_state=0, max_iter=90):
    x_tr, x_te, y_tr, y_te = split_scale(X, y)
    clf = MLPClassifier(hidden_layer_sizes=(16,), activation="relu", solver="adam", alpha=0.001, max_iter=max_iter, random_state=random_state)
    clf.fit(x_tr, y_tr)
    proba = clf.predict_proba(x_te)
    loss = log_loss(y_te, proba, labels=clf.classes_)
    acc = accuracy_score(y_te, clf.predict(x_te))
    return float(loss), float(acc), clf


def train_small_mlp_acc(X, y, random_state=0, max_iter=90):
    loss, acc, clf = train_small_mlp_loss(X, y, random_state=random_state, max_iter=max_iter)
    return acc, loss, clf


def preview_ladder(rungs):
    for rung, (name, X, y) in enumerate(rungs, 1):
        classes = np.unique(y)
        print(f"D{rung}: {name:36s} X={X.shape} classes={len(classes)} sample_y={classes[:5].tolist()}")


def plot_metric_table(rows, metric_name):
    print(f"{'rung':<4} {'dataset':<36} {metric_name:>10}")
    for row in rows:
        print(f"{row['rung']:<4} {row['name']:<36} {row['metric']:10.4f}")


def plot_summary(rows, metric_name, ylabel):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    names = [row["rung"] for row in rows]
    values = [row["metric"] for row in rows]
    axes[0].bar(names, values, color="steelblue")
    axes[0].set_title("Metric by ladder rung")
    axes[0].set_ylabel(ylabel)
    axes[1].plot(names, values, marker="o")
    axes[1].set_title("D1→D5 trend")
    axes[1].set_ylabel(ylabel)
    fig.tight_layout()
    plt.show()


def show_artifacts(rungs, title):
    fig, axes = plt.subplots(1, len(rungs), figsize=(13, 2.6))
    for ax, (name, X, y) in zip(axes, rungs):
        if X.shape[1] == 64:
            ax.imshow(X[0].reshape(8, 8), cmap="gray")
            ax.set_title(name.split()[0])
        else:
            ax.scatter(X[:, 0], X[:, 1], c=y, s=12, cmap="tab10")
            ax.set_title(name.split()[0])
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


## The concept, built once

Fine-tuning uses separate updates: $\theta_{base}\leftarrow\theta_{base}-\eta_{base}g_{base}$ and $\theta_{head}\leftarrow\theta_{head}-\eta_{head}g_{head}$. The lesson scalar update is $2.0-0.09\cdot1.65=1.8515$.

In [ ]:

def fine_tune_update(base, head, g_base, g_head, lr_base, lr_head):
    new_base = base - lr_base * g_base
    new_head = head - lr_head * g_head
    return new_base, new_head

base, head = fine_tune_update(2.0, 2.0, 1.65, 1.65, 0.009, 0.09)
lesson_update = 2.0 - 0.09 * 1.65
print("base update", base)
print("head update", head)
print("lesson head update", lesson_update)
assert np.isclose(head, 1.8515)
assert abs(2.0 - base) < abs(2.0 - head)


We pretrain a real hidden base on D3 noisy moons after zero-padding to a common 64-D input, then transfer the base to each rung with a fresh head.

In [ ]:

def one_hot(y, classes):
    mapper = {label: i for i, label in enumerate(classes)}
    return np.eye(len(classes))[[mapper[v] for v in y]]


def relu(Z):
    return np.maximum(0.0, Z)


def train_two_layer(X, y, W1=None, W2=None, freeze_epochs=20, tune_epochs=40, lr_base=0.01, lr_head=0.08):
    X64 = pad_to_64(X)
    x_tr, x_te, y_tr, y_te = split_scale(X64, y)
    classes = np.unique(y_tr)
    Y = one_hot(y_tr, classes)
    rng = np.random.default_rng(24)
    if W1 is None:
        W1 = rng.normal(0.0, 0.15, size=(x_tr.shape[1], 18))
    else:
        W1 = W1.copy()
    if W2 is None or W2.shape[1] != len(classes):
        W2 = rng.normal(0.0, 0.15, size=(18, len(classes)))
    for epoch in range(freeze_epochs + tune_epochs):
        H = relu(x_tr @ W1)
        P = softmax_logits(H, W2)
        dlogits = (P - Y) / len(x_tr)
        gW2 = H.T @ dlogits
        gH = dlogits @ W2.T
        gW1 = x_tr.T @ (gH * (H > 0))
        W2 = W2 - lr_head * gW2
        if epoch >= freeze_epochs:
            W1 = W1 - lr_base * gW1
    Pte = softmax_logits(relu(x_te @ W1), W2)
    preds = classes[np.argmax(Pte, axis=1)]
    return float(accuracy_score(y_te, preds)), W1, W2

source_name, source_X, source_y = clf_digits_ladder()[2]
source_acc, pretrained_W1, source_W2 = train_two_layer(source_X, source_y, freeze_epochs=0, tune_epochs=80, lr_base=0.03, lr_head=0.08)
print("D3 pretrain source accuracy", source_acc)


## The dataset ladder

The same code is run from a four-point XOR problem through real 8×8 digit images and a noisy shifted D5 variant.

In [ ]:
rungs = clf_digits_ladder()
preview_ladder(rungs)
show_artifacts(rungs, 'Ladder preview')

## Run the SAME method across D1–D5

In [ ]:

rows = []
for idx, (name, X, y) in enumerate(rungs, 1):
    if idx >= 4:
        acc, W1, W2 = train_two_layer(X, y, W1=pretrained_W1, freeze_epochs=25, tune_epochs=35, lr_base=0.006, lr_head=0.07)
    else:
        acc, W1, W2 = train_two_layer(X, y, W1=pretrained_W1, freeze_epochs=5, tune_epochs=35, lr_base=0.01, lr_head=0.07)
    rows.append({"rung": f"D{idx}", "name": name, "metric": acc})
plot_metric_table(rows, "accuracy")


## Results visualization

In [ ]:
show_artifacts(rungs, 'Transfer-learning artifacts')
plot_summary(rows, 'accuracy', 'held-out accuracy')

## Pitfall on D5: overfitting a few labels with full fine-tuning

In [ ]:

name, X, y = rungs[-1]
slow_acc, W1, W2 = train_two_layer(X, y, W1=pretrained_W1, freeze_epochs=25, tune_epochs=35, lr_base=0.004, lr_head=0.07)
fast_acc, W1b, W2b = train_two_layer(X, y, W1=pretrained_W1, freeze_epochs=0, tune_epochs=60, lr_base=0.8, lr_head=0.8)
print("freeze then slow unfreeze", slow_acc)
print("full fast fine-tune", fast_acc)
assert slow_acc >= fast_acc - 0.20


## Evaluate it + Practice

- Metric: held-out accuracy; compare it with a majority-class or plain logistic baseline.
- Sanity check: D1 should be hand-checkable before trusting D5.
- Ablation: turn off the key idea and the reported metric should get worse or the pitfall should reappear.
- Failure signals: unstable losses, collapsed routing, inflated gradient error, or a D5 result that ignores label noise.

Practice 1: change one hyperparameter and rerun the D1 assert before touching D5.

Practice 2: add a baseline row to the ladder table and explain the largest gap.

Practice 3: make the D5 pitfall more severe, then show the same fix still helps.